# Interfaces

## A Protocol and Two Classes

Define the following protocol and classes:

- The protocol is called `Aggregator` and asks for a method `reduce(self, items: list[int]) -> float`
- `Sum`, which inherits from nothing, implements `reduce` by summing the items
- `Minimum`, which inherits from nothing either, implements `reduce` by finding the minimum. It accepts a `key` parameter in its `__init__` method

Test this code with a list you create.

In [ ]:
# Your code here

### Additional hints

The body of a protocol's method is `...`. `Sum` and `Minimum` are ordinary classes, with no parent: they conform to the protocol through their `reduce` method alone. `Minimum` keeps `key` in `self.key`, and passes it to `min(items, key=self.key)`.

With `data = [1, 2, 3]`, `Minimum().reduce(data)` returns `1`, `Sum().reduce(data)` returns `6`, and `Minimum(key=lambda x: -x).reduce(data)` returns `3`.

```python
from typing import Protocol


class Aggregator(Protocol):
  def reduce(self, items: list[int]) -> float: ...


class Sum:
  def reduce(self, items: list[int]) -> float: ...


class Minimum:
  def __init__(self, key=None): ...

  def reduce(self, items: list[int]) -> float: ...
```

### Solution

In [ ]:
from typing import Protocol


class Aggregator(Protocol):
  def reduce(self, items: list[int]) -> float: ...


class Sum:
  def reduce(self, items: list[int]) -> float:
    return sum(items)


class Minimum:
  def __init__(self, key=None):
    self.key = key

  def reduce(self, items: list[int]) -> float:
    return min(items, key=self.key)


data = [1, 2, 3]
print(Minimum().reduce(data))
print(Sum().reduce(data))
print(Minimum(key=lambda x: -x).reduce(data))

## Coding to the Protocol

Write a client function `summarize(aggregator: Aggregator, values: list[int]) -> str` that returns a text like `"3 values: 6"`: the number of values, then the result of `reduce`. It must only know the protocol, neither `Sum` nor `Minimum`.

Call it with `Sum()` and `Minimum()`. Then add a third class, `Mean`, and check that `summarize` works without touching it.

In [ ]:
# Your code here

### Additional hints

`summarize` only calls `aggregator.reduce(values)`, and `len(values)` for the number of values. `Mean`'s `reduce` method divides the sum by the number of items.

With `data = [1, 2, 3]`, `summarize(Mean(), data)` returns `"3 values: 2.0"`.

```python
def summarize(aggregator: Aggregator, values: list[int]) -> str:
  return f"{len(values)} values: {...}"


class Mean:
  def reduce(self, items: list[int]) -> float: ...
```

### Solution

In [ ]:
def summarize(aggregator: Aggregator, values: list[int]) -> str:
  return f"{len(values)} values: {aggregator.reduce(values)}"


class Mean:
  def reduce(self, items: list[int]) -> float:
    return sum(items) / len(items)


print(summarize(Sum(), data))
print(summarize(Minimum(), data))
print(summarize(Mean(), data))

## What Python Doesn't Check

What can you say about the following class? What happens when calling `summarize(Median(), data)`?

In [ ]:
class Median:
  def reduction(self, items: list[int]) -> float:
    return sorted(items)[len(items) // 2]

In [ ]:
# Your code here

### Additional hints

`Median` has `reduction`, not `reduce`. The call raises an exception: catch it, so that the notebook goes on.

```python
try:
  summarize(Median(), data)
except AttributeError as error:
  ...
```

### Solution

In [ ]:
# Python doesn't look at the Aggregator annotation: the error only comes when reduce
# is called, in summarize, and nothing announces it before.
try:
  summarize(Median(), data)
except AttributeError as error:
  print("AttributeError:", error)

## The Type Checker

The next cells install `pyrefly` and write this code to a file. Run `pyrefly` on that file, read its error, then fix `Median` until it reports nothing.

**Useful documentation**

- [Pyrefly, configuration](https://pyrefly.org/en/docs/configuration/)

In [ ]:
!pip install -q pyrefly

In [ ]:
%%writefile aggregators.py
from typing import Protocol


class Aggregator(Protocol):
  def reduce(self, items: list[int]) -> float: ...


class Median:
  def reduction(self, items: list[int]) -> float:
    return sorted(items)[len(items) // 2]


def summarize(aggregator: Aggregator, values: list[int]) -> str:
  return f"{len(values)} values: {aggregator.reduce(values)}"


print(summarize(Median(), [1, 2, 3]))

In [ ]:
# Your code here

### Additional hints

Without a configuration file, `pyrefly` applies a minimal preset that says nothing here: pass it `--preset default`. It reports the line of the call to `summarize`, before anything runs. Fix it by renaming `reduction`, with `%%writefile` or in the editor, then run it again.

```python
!pyrefly check --preset default aggregators.py
```

### Solution

In [ ]:
!pyrefly check --preset default aggregators.py

In [ ]:
%%writefile aggregators.py
from typing import Protocol


class Aggregator(Protocol):
  def reduce(self, items: list[int]) -> float: ...


class Median:
  def reduce(self, items: list[int]) -> float:
    return sorted(items)[len(items) // 2]


def summarize(aggregator: Aggregator, values: list[int]) -> str:
  return f"{len(values)} values: {aggregator.reduce(values)}"


print(summarize(Median(), [1, 2, 3]))

In [ ]:
!pyrefly check --preset default aggregators.py

## A Class You Can't Change

Define a `Closable` protocol, which asks for a `close(self) -> None` method, and a function `close_all(objects: list[Closable]) -> None` that closes them all.

Call it with an `io.StringIO` and an open file: their classes come from the standard library, and don't know your protocol. Print their `closed` attribute to check.

In [ ]:
# Your code here

### Additional hints

`close_all` loops over the list and calls `close` on each object. An open file: `open("notes.txt", "w", encoding="utf-8")`. After the call, `closed` is `True` for both.

```python
from io import StringIO
from typing import Protocol


class Closable(Protocol):
  def close(self) -> None: ...


def close_all(objects: list[Closable]) -> None: ...
```

### Solution

In [ ]:
from io import StringIO


class Closable(Protocol):
  def close(self) -> None: ...


def close_all(objects: list[Closable]) -> None:
  for obj in objects:
    obj.close()


buffer = StringIO("some text")
file = open("notes.txt", "w", encoding="utf-8")
close_all([buffer, file])
print(buffer.closed, file.closed)

## `@runtime_checkable` and Its Limits

Consider the following class:

In [ ]:
class Draft:
  def reduce(self) -> float:
    return 0.0

What does `isinstance(Sum(), Aggregator)` give?

Then define `CheckableAggregator`, the same protocol decorated with `@runtime_checkable`, and compare `isinstance` on `Sum()`, `Median()` and `Draft()`. What doesn't `isinstance` check?

In [ ]:
# Your code here

### Additional hints

Without the decorator, `isinstance` raises a `TypeError`, to catch. With it, you should get `True`, `False`, then `True`: `Draft` does have a `reduce` method, but with no parameter.

```python
from typing import runtime_checkable


@runtime_checkable
class CheckableAggregator(Protocol):
  def reduce(self, items: list[int]) -> float: ...
```

### Solution

In [ ]:
from typing import runtime_checkable

try:
  isinstance(Sum(), Aggregator)
except TypeError as error:
  print("TypeError:", error)


@runtime_checkable
class CheckableAggregator(Protocol):
  def reduce(self, items: list[int]) -> float: ...


print(isinstance(Sum(), CheckableAggregator))
print(isinstance(Median(), CheckableAggregator))

# Draft passes: isinstance looks at whether the method exists, not at its signature.
# Only the type checker sees that its reduce takes no parameter.
print(isinstance(Draft(), CheckableAggregator))

## A Standard Library Protocol (Optional)

`collections.abc` offers ready-made protocols. Write a `Basket` class that has a `__len__` method, with no parent, and a function `is_empty(obj: Sized) -> bool`. Try it with `Basket` and a list, then check `isinstance(Basket([]), Sized)`.

**Useful documentation**

- [`collections.abc.Sized`](https://docs.python.org/3/library/collections.abc.html#collections.abc.Sized)

In [ ]:
# Your code here

### Additional hints

`Sized` is imported from `collections.abc`. It accepts `isinstance` with no decorator: `isinstance(Basket([]), Sized)` returns `True`, with no inheritance.

```python
from collections.abc import Sized


class Basket:
  def __init__(self, products: list[str]) -> None:
    self.products = products

  def __len__(self) -> int: ...


def is_empty(obj: Sized) -> bool: ...
```

### Solution

In [ ]:
from collections.abc import Sized


class Basket:
  def __init__(self, products: list[str]) -> None:
    self.products = products

  def __len__(self) -> int:
    return len(self.products)


def is_empty(obj: Sized) -> bool:
  return len(obj) == 0


print(is_empty(Basket([])), is_empty([1]))
print(isinstance(Basket([]), Sized))